In [2]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)

perf_df = pd.read_csv("dec-jan.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [25]:
import numpy as np
import pandas as pd

def add_streak_cols(df, ret_col, *, date_col="Date"):
    d = df[[date_col, "Close", ret_col]].sort_values(date_col).copy()

    s = d[ret_col].astype("int8")
    grp = s.ne(s.shift()).cumsum()
    streak_len = s.groupby(grp).cumcount() + 1

    d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
    d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
    return d

def streak_perf_tables(df_daily, perf_df, returns, *, test_days=1):
    out_by_r = {}
    side_by_r = {}

    base = df_daily[["Date", "Close"] + [f"Return_{r}" for r in returns]].copy()

    for r in returns:
        ret_col = f"Return_{r}"

        # streaks for this horizon
        df_r = add_streak_cols(base, ret_col)

        # merge with performance rows for this horizon/test_days
        perf_r = perf_df[(perf_df["horizon"] == r) & (perf_df["test_days"] == test_days)]
        dfm = df_r.merge(perf_r, on="Date", how="inner")

        gcols = ["model", "train_years", "feature_set"]

        # 1) Accuracy by streak_lag1
        acc_piv = dfm.pivot_table(
            index=gcols,
            columns="streak_lag1",
            values="acc",
            aggfunc="mean",
        )

        # 2) Count by streak_lag1
        cnt_piv = dfm.pivot_table(
            index=gcols,
            columns="streak_lag1",
            values="acc",
            aggfunc="size",
        )

        out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

        # sort streak columns (negative first, then 0, then positive)
        out = out.reindex(
            columns=sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])),
        )

        out_by_r[r] = out

        # pos/neg summary
        df2 = dfm.copy()
        df2["side"] = np.where(df2["streak"] > 0, "pos", "neg")

        side_perf = (
            df2.groupby(gcols + ["side"], sort=False)
               .agg(n=("acc", "size"), acc=("acc", "mean"))
               .reset_index()
        )

        side_wide = (
            side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
                     .round(2)
        )

        side_by_r[r] = side_wide

    # optional: one combined object with horizon as an extra index level
    all_out = pd.concat(out_by_r, names=["horizon"])
    all_side = pd.concat(side_by_r, names=["horizon"])

    return out_by_r, side_by_r, all_out, all_side

returns = [2, 5, 10]
out_by_r, side_by_r, all_out, all_side = streak_perf_tables(df_daily, perf_df, returns, test_days=1)

# example: horizon 10 tables
out_by_r[10]
side_by_r[10]

# or combined across horizons
#all_out
all_side


acc           n      
side                                            neg   pos   neg   pos
horizon model         train_years feature_set                        
2       random_forest 4           daily        0.50  0.70  22.0  23.0
                      6           daily        0.50  0.78  22.0  23.0
        xgboost       4           daily        0.41  0.70  22.0  23.0
                      6           daily        0.41  0.65  22.0  23.0
5       random_forest 4           daily        0.72  0.93  18.0  27.0
                      6           daily        0.67  0.89  18.0  27.0
        xgboost       4           daily        0.61  0.85  18.0  27.0
                      6           daily        0.44  0.89  18.0  27.0
10      random_forest 4           daily        0.86  0.74  22.0  23.0
                      6           daily        0.82  0.78  22.0  23.0
        xgboost       4           daily        0.82  0.87  22.0  23.0
                      6           daily        0.82  0.83  22.0  23.0

In [27]:
import numpy as np
import pandas as pd

def flip_bucket_tables_multi(
    df_daily,
    perf_df,
    returns,
    *,
    K=3,
    test_days=1,
    date_col="Date",
    close_col="Close",
    streak_col="streak_lag1",
    w=None,
):
    """
    For each horizon r:
      - builds streak_lag1 from Return_r (via day-to-day streak logic)
      - merges into perf_df rows for (horizon=r, test_days)
      - buckets streak_lag1 into [-K..K] plus tails as +/- (K+1) labeled "3+"
      - computes acc/n wide table and weighted balanced-accuracy score (wba)

    Returns:
      flip_by_r: dict[r] -> flip_wide (MultiIndex columns)
      rf_sorted_by_r: dict[r] -> sorted flip_wide by wba
      all_flip: concat of flip_wide with horizon as index level
    """
    if w is None:
        # weights: ±1 -> 2.0, ±2 -> 1.5, ±3 -> 1.25, ±3+ -> 1.0
        w = {1: 2.0, 2: 1.5, 3: 1.25, "3+": 1.0}

    max_score = sum(w.values())
    gcols = ["model", "train_years", "feature_set"]

    # --- helper: compute streak + streak_lag1 for a given Return_r ---
    def _add_streak(df_base, ret_col):
        d = df_base[[date_col, close_col, ret_col]].sort_values(date_col).copy()
        s = d[ret_col].astype("int8")
        grp = s.ne(s.shift()).cumsum()
        streak_len = s.groupby(grp).cumcount() + 1
        d["streak"] = streak_len.where(s.eq(1), -streak_len).astype("int32")
        d["streak_lag1"] = d["streak"].shift(1).fillna(0).astype("Int64")
        return d

    base = df_daily[[date_col, close_col] + [f"Return_{r}" for r in returns]].copy()

    flip_by_r = {}
    rf_sorted_by_r = {}

    for r in returns:
        ret_col = f"Return_{r}"

        # build streak_lag1
        df_r = _add_streak(base, ret_col)

        # merge with perf (this gives you acc/model/train_years/feature_set/etc)
        perf_r = perf_df[(perf_df["horizon"] == r) & (perf_df["test_days"] == test_days)]
        d = df_r.merge(perf_r, on=date_col, how="inner")

        # --- bucket streak_lag1 into [-K..K] plus tails as +/- (K+1) ---
        d["streak_bucket"] = d[streak_col].clip(lower=-K, upper=K)
        d.loc[d[streak_col] < -K, "streak_bucket"] = -(K + 1)
        d.loc[d[streak_col] >  K, "streak_bucket"] =  (K + 1)

        flip_perf = (
            d.groupby(gcols + ["streak_bucket"], sort=False)
             .agg(n=("acc", "size"), acc=("acc", "mean"))
        )

        flip_wide = pd.concat(
            {"acc": flip_perf["acc"].unstack("streak_bucket"),
             "n":   flip_perf["n"].unstack("streak_bucket")},
            axis=1
        )

        # enforce column order: +1/-1, +2/-2, +3/-3, 3+/-3+
        ordered_cols = []
        for k in [1, 2, 3, "3+"]:
            pb = (K + 1) if k == "3+" else k
            nb = -(K + 1) if k == "3+" else -k
            ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]
        flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

        # relabel buckets to strings ("3+", "-3+", etc.)
        rename_cols = []
        for metric, b in flip_wide.columns:
            if b == (K + 1): lab = "3+"
            elif b == -(K + 1): lab = "-3+"
            else: lab = str(b)
            rename_cols.append((metric, lab))
        flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

        # --- per-pair bal_acc and weighted score ---
        def _pair_bal(pos_lab, neg_lab):
            return (flip_wide[("acc", pos_lab)] + flip_wide[("acc", neg_lab)]) / 2

        flip_wide[("bal_acc_pair", "1")]  = _pair_bal("1",  "-1")
        flip_wide[("bal_acc_pair", "2")]  = _pair_bal("2",  "-2")
        flip_wide[("bal_acc_pair", "3")]  = _pair_bal("3",  "-3")
        flip_wide[("bal_acc_pair", "3+")] = _pair_bal("3+", "-3+")

        # weighted SUM, normalized by max_score (your current behavior)
        flip_wide[("wba", "")] = (
            w[1]    * flip_wide[("bal_acc_pair", "1")] +
            w[2]    * flip_wide[("bal_acc_pair", "2")] +
            w[3]    * flip_wide[("bal_acc_pair", "3")] +
            w["3+"] * flip_wide[("bal_acc_pair", "3+")]
        ) / max_score

        flip_wide[("wba", "")] = flip_wide[("wba", "")].round(2)

        flip_by_r[r] = flip_wide
        rf_sorted_by_r[r] = flip_wide.sort_values(by=("wba", ""), ascending=False).round(2)

    all_flip = pd.concat(flip_by_r, names=["horizon"])
    return flip_by_r, rf_sorted_by_r, all_flip

returns = [2, 5, 10]
flip_by_r, rf_sorted_by_r, all_flip = flip_bucket_tables_multi(
    df_daily=df_daily,
    perf_df=perf_df,
    returns=returns,
    K=3,
    test_days=1,
)

rf_sorted_by_r[10]     # horizon 10 sorted table
flip_by_r[5]           # horizon 5 unsorted table
all_flip.round(2)               # all horizons stacked with "horizon" index level


acc        n      acc        \
                                                  1    -1  1 -1     2    -2   
horizon model         train_years feature_set                                 
2       xgboost       4           daily        0.67  0.00  6  6  0.40  0.50   
                      6           daily        0.83  0.33  6  6  0.40  0.75   
        random_forest 4           daily        0.50  0.33  6  6  0.40  0.75   
                      6           daily        0.67  0.50  6  6  0.40  0.50   
5       xgboost       4           daily        0.75  1.00  4  4  0.67  0.67   
                      6           daily        0.75  0.50  4  4  1.00  0.67   
        random_forest 4           daily        1.00  0.75  4  4  1.00  1.00   
                      6           daily        0.75  0.75  4  4  1.00  1.00   
10      xgboost       4           daily        0.67  0.67  3  3  1.00  1.00   
                      6           daily        1.00  0.33  3  3  1.00  1.00   
        random_forest 4           daily        0.67  0.67  3  3  1.00  1.00   
                      6           daily        0.67  0.33  3  3  1.00  1.00   

                                               n      acc        ...  n   acc  \
                                               2 -2     3    -3  ... -3    3+   
horizon model         train_years feature_set                    ...            
2       xgboost       4           daily        5  4  0.67  0.75  ...  4  0.78   
                      6           daily        5  4  0.67  0.25  ...  4  0.56   
        random_forest 4           daily        5  4  0.67  0.75  ...  4  0.78   
                      6           daily        5  4  0.67  0.75  ...  4  0.78   
5       xgboost       4           daily        3  3  0.67  0.33  ...  3  0.82   
                      6           daily        3  3  1.00  0.00  ...  3  0.82   
        random_forest 4           daily        3  3  1.00  0.33  ...  3  0.82   
                      6           daily        3  3  1.00  0.33  ...  3  0.82   
10      xgboost       4           daily        3  2  1.00  1.00  ...  2  0.79   
                      6           daily        3  2  1.00  1.00  ...  2  0.71   
        random_forest 4           daily        3  2  1.00  1.00  ...  2  0.64   
                      6           daily        3  2  1.00  1.00  ...  2  0.71   

                                                      n     bal_acc_pair  \
                                                -3+  3+ -3+            1   
horizon model         train_years feature_set                              
2       xgboost       4           daily        0.62   9   8         0.33   
                      6           daily        0.50   9   8         0.58   
        random_forest 4           daily        0.62   9   8         0.42   
                      6           daily        0.75   9   8         0.58   
5       xgboost       4           daily        0.75  17   8         0.88   
                      6           daily        0.62  17   8         0.62   
        random_forest 4           daily        0.88  17   8         0.88   
                      6           daily        0.75  17   8         0.75   
10      xgboost       4           daily        0.87  14  15         0.67   
                      6           daily        0.87  14  15         0.67   
        random_forest 4           daily        0.87  14  15         0.67   
                      6           daily        0.87  14  15         0.50   

                                                                  wba  
                                                  2     3    3+        
horizon model         train_years feature_set                          
2       xgboost       4           daily        0.45  0.71  0.70  0.51  
                      6           daily        0.57  0.46  0.53  0.54  
        random_forest 4           daily        0.57  0.71  0.70  0.57  
                      6           daily        0.45  0.71  0.76  0.61  
5